In [1]:
import pandas as pd
import numpy as np

metadata = pd.read_csv("metaml_metadata_final.csv")
mtb = pd.read_csv("metaml_species_abundance.csv")

feature_cols = [c for c in mtb.columns if c != "Sample"]
X = mtb[feature_cols].fillna(0)
y = metadata["Study.Group"]

print("X shape:", X.shape)
print(y.value_counts())

X shape: (445, 3302)
Study.Group
Control    272
UC         148
CD          25
Name: count, dtype: int64


In [2]:
sparsity = (X == 0).sum(axis=0) / len(X) * 100
X_filtered = X[sparsity[sparsity <= 80].index]

print("Features before:", X.shape[1])
print("Features after sparsity filter:", X_filtered.shape[1])

Features before: 3302
Features after sparsity filter: 407


In [3]:
def clr_transform(data, pseudocount=1):
    data_p1 = data + pseudocount
    geo_mean = np.exp(np.log(data_p1).mean(axis=1))
    return np.log(data_p1.div(geo_mean, axis=0))

X_clr = clr_transform(X_filtered)
print("CLR transform done. Shape:", X_clr.shape)

CLR transform done. Shape: (445, 407)


In [4]:
from sklearn.ensemble import IsolationForest

iso = IsolationForest(contamination=0.05, random_state=42)
outlier_flags = iso.fit_predict(X_clr)

print("Outliers detected:", (outlier_flags == -1).sum(), "of", len(X_clr))

Outliers detected: 23 of 445


In [5]:
X_clean = X_clr[outlier_flags == 1].reset_index(drop=True)
y_clean = y[outlier_flags == 1].reset_index(drop=True)

print("Samples after outlier removal:", len(X_clean))
print(y_clean.value_counts())

Samples after outlier removal: 422
Study.Group
Control    268
UC         132
CD          22
Name: count, dtype: int64


In [6]:
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

y_binary = y_clean.map(lambda g: "Control" if g == "Control" else "IBD")
ibd_mask = (y_binary == "IBD").values
control_mask = (y_binary == "Control").values

pvals = {}
for col in X_clean.columns:
    try:
        stat, p = mannwhitneyu(X_clean.loc[ibd_mask, col], X_clean.loc[control_mask, col])
    except ValueError:
        p = 1.0
    pvals[col] = p

pval_series = pd.Series(pvals)
fdr_corrected = multipletests(pval_series.values, method="fdr_bh")[1]
fdr_series = pd.Series(fdr_corrected, index=pval_series.index)

significant_features = fdr_series[fdr_series < 0.05].index.tolist()
print("Significant species at FDR < 0.05:", len(significant_features), "of", len(X_clean.columns))

Significant species at FDR < 0.05: 270 of 407


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

X_final = X_clean[significant_features]
le = LabelEncoder()
y_enc = le.fit_transform(y_clean)

train_idx, test_idx = train_test_split(np.arange(len(y_enc)), test_size=0.2, stratify=y_enc, random_state=42)
y_train, y_test = y_enc[train_idx], y_enc[test_idx]

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_final.iloc[train_idx]), columns=X_final.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_final.iloc[test_idx]), columns=X_final.columns)

rf = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42)
rf.fit(X_train_scaled, y_train)
rf_auc = roc_auc_score(y_test, rf.predict_proba(X_test_scaled), multi_class="ovr", average="macro")

xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, eval_metric="mlogloss")
xgb.fit(X_train_scaled, y_train)
xgb_auc = roc_auc_score(y_test, xgb.predict_proba(X_test_scaled), multi_class="ovr", average="macro")

print(f"Random Forest — macro-AUC: {rf_auc:.3f}")
print(f"XGBoost — macro-AUC: {xgb_auc:.3f}")
print(f"Classes: {le.classes_}")
print(f"Test set size: {len(y_test)}")

Random Forest — macro-AUC: 0.957
XGBoost — macro-AUC: 0.964
Classes: ['CD' 'Control' 'UC']
Test set size: 85


In [8]:
from scipy.stats import kruskal

# IBDpred uses log2 transform (not CLR) on the sparsity-filtered, outlier-cleaned data
X_ibd_clean = X_filtered[outlier_flags == 1].reset_index(drop=True)
y_ibd_clean = y[outlier_flags == 1].reset_index(drop=True)
X_log = np.log2(X_ibd_clean + 1)

def kruskal_top_k(Xd, yd, k):
    groups = pd.Series(yd).unique()
    pvals = {c: kruskal(*[Xd.loc[pd.Series(yd).values == g, c] for g in groups])[1] for c in Xd.columns}
    return sorted(pvals, key=pvals.get)[:k]

train_idx2, test_idx2 = train_test_split(np.arange(len(y_ibd_clean)), test_size=0.2, stratify=y_ibd_clean, random_state=42)

top40_metaml = kruskal_top_k(X_log.iloc[train_idx2], y_ibd_clean.iloc[train_idx2], 40)
print("Top 40 features selected")

Top 40 features selected


In [9]:
from sklearn.linear_model import LogisticRegression

X_train_ibd = X_log[top40_metaml].iloc[train_idx2]
X_test_ibd = X_log[top40_metaml].iloc[test_idx2]

le_ibd = LabelEncoder()
y_ibd_enc = le_ibd.fit_transform(y_ibd_clean)
y_train_ibd, y_test_ibd = y_ibd_enc[train_idx2], y_ibd_enc[test_idx2]

rf_ibdpred = RandomForestClassifier(n_estimators=500, max_features="sqrt", random_state=42)
rf_ibdpred.fit(X_train_ibd, y_train_ibd)
rf_ibd_auc = roc_auc_score(y_test_ibd, rf_ibdpred.predict_proba(X_test_ibd), multi_class="ovr", average="macro")

X_train_ibd_scaled = StandardScaler().fit_transform(X_train_ibd)
X_test_ibd_scaled = StandardScaler().fit(X_train_ibd).transform(X_test_ibd)
en_ibdpred = LogisticRegression(penalty="elasticnet", solver="saga", l1_ratio=0.5, C=1, max_iter=20000, random_state=42)
en_ibdpred.fit(X_train_ibd_scaled, y_train_ibd)
en_ibd_auc = roc_auc_score(y_test_ibd, en_ibdpred.predict_proba(X_test_ibd_scaled), multi_class="ovr", average="macro")

print(f"IBDpred-style Random Forest — macro-AUC: {rf_ibd_auc:.3f}")
print(f"IBDpred-style Elastic Net — macro-AUC: {en_ibd_auc:.3f}")

IBDpred-style Random Forest — macro-AUC: 0.926
IBDpred-style Elastic Net — macro-AUC: 0.933


In [10]:
prevalence = (X_ibd_clean > 0).mean(axis=0)
keep_siamcat = prevalence[prevalence >= 0.10].index
feat_log = np.log10(X_ibd_clean[keep_siamcat] + 1e-6)
feat_norm = feat_log.div(feat_log.sum(axis=1), axis=0)

def auc_feature_ranking(Xd, yd, n_features=100):
    scores = {}
    for col in Xd.columns:
        try:
            score = roc_auc_score(yd, Xd[col], multi_class="ovr")
        except ValueError:
            score = 0.5
        scores[col] = abs(score - 0.5)
    ranked = sorted(scores, key=scores.get, reverse=True)
    return ranked[:n_features]

y_binary_train = pd.Series(y_ibd_clean).map(lambda g: 0 if g == "Control" else 1).iloc[train_idx2]
top100_metaml = auc_feature_ranking(feat_norm.iloc[train_idx2], y_binary_train, n_features=100)

X_train_s = feat_norm[top100_metaml].iloc[train_idx2]
X_test_s = feat_norm[top100_metaml].iloc[test_idx2]

rf_siamcat = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)
rf_siamcat.fit(X_train_s, y_train_ibd)
siamcat_auc = roc_auc_score(y_test_ibd, rf_siamcat.predict_proba(X_test_s), multi_class="ovr", average="macro")

print(f"SIAMCAT-style Random Forest — macro-AUC: {siamcat_auc:.3f}")

SIAMCAT-style Random Forest — macro-AUC: 0.954
